# 1교시 · 공공·국제 오픈데이터와 API의 이해

**이 시간에 할 일**

1. 인증 없이 국제기구 API를 한 번 호출해 봅니다
2. 응답이 어떤 구조인지 직접 뜯어봅니다
3. 우리 기관 데이터 소스를 수집 가능성 기준으로 점검합니다

---

**참여 방법 두 가지**

- **따라하기** — 셀을 위에서부터 순서대로 실행합니다
- **관전** — 화면만 보시고, 나중에 이 노트북을 각자 실행하셔도 됩니다

인터넷이 막혀 있거나 호출이 실패해도 아래 `OFFLINE = True`로 바꾸면 내장 샘플로 전부 따라오실 수 있습니다.

In [ ]:
# ── 설정 ───────────────────────────────────────────────
OFFLINE = False   # 호출이 안 되면 True 로 바꾸고 다시 실행하세요

import json
import pandas as pd
import requests

pd.set_option("display.max_rows", 30)
print("준비 완료")

## 1-1. 인증 없이 첫 호출

World Bank Indicators API는 인증키가 필요 없습니다. 주소 하나로 끝납니다.

```
https://api.worldbank.org/v2/country/{국가}/indicator/{지표}?format=json
```

- `국가` — ISO 3166-1 alpha-3 코드 (한국은 `KOR`)
- `지표` — World Bank 지표 코드 (합계출산율은 `SP.DYN.TFRT.IN`)
- `format=json` — 이걸 빼면 XML이 옵니다

In [ ]:
# 내장 샘플 (오프라인용)
SAMPLE_WB = [
    {"page": 1, "pages": 1, "per_page": 50, "total": 6,
     "sourceid": "2", "lastupdated": "2026-07-01"},
    [
        {"indicator": {"id": "SP.DYN.TFRT.IN", "value": "Fertility rate, total (births per woman)"},
         "country": {"id": "KR", "value": "Korea, Rep."},
         "countryiso3code": "KOR", "date": "2024", "value": 0.75, "unit": "", "obs_status": "", "decimal": 1},
        {"indicator": {"id": "SP.DYN.TFRT.IN", "value": "Fertility rate, total (births per woman)"},
         "country": {"id": "KR", "value": "Korea, Rep."},
         "countryiso3code": "KOR", "date": "2023", "value": 0.72, "unit": "", "obs_status": "", "decimal": 1},
        {"indicator": {"id": "SP.DYN.TFRT.IN", "value": "Fertility rate, total (births per woman)"},
         "country": {"id": "KR", "value": "Korea, Rep."},
         "countryiso3code": "KOR", "date": "2022", "value": 0.78, "unit": "", "obs_status": "", "decimal": 1},
        {"indicator": {"id": "SP.DYN.TFRT.IN", "value": "Fertility rate, total (births per woman)"},
         "country": {"id": "KR", "value": "Korea, Rep."},
         "countryiso3code": "KOR", "date": "2021", "value": 0.81, "unit": "", "obs_status": "", "decimal": 1},
        {"indicator": {"id": "SP.DYN.TFRT.IN", "value": "Fertility rate, total (births per woman)"},
         "country": {"id": "KR", "value": "Korea, Rep."},
         "countryiso3code": "KOR", "date": "2020", "value": 0.84, "unit": "", "obs_status": "", "decimal": 1},
        {"indicator": {"id": "SP.DYN.TFRT.IN", "value": "Fertility rate, total (births per woman)"},
         "country": {"id": "KR", "value": "Korea, Rep."},
         "countryiso3code": "KOR", "date": "2019", "value": None, "unit": "", "obs_status": "", "decimal": 1},
    ],
]


def wb_call(country="KOR", indicator="SP.DYN.TFRT.IN", start=2010, end=2025):
    """World Bank Indicators API 호출. 실패하면 내장 샘플을 돌려줍니다."""
    url = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"
    params = {"format": "json", "date": f"{start}:{end}", "per_page": 500}
    if OFFLINE:
        return SAMPLE_WB
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"[호출 실패] {type(e).__name__}: {e}")
        print("→ 내장 샘플로 대체합니다. 이후 셀은 그대로 진행됩니다.")
        return SAMPLE_WB


payload = wb_call()
print(type(payload), "길이", len(payload))

### 응답 구조를 먼저 봅니다

여기서 첫 번째 함정이 나옵니다. 응답 최상위가 **딕셔너리가 아니라 원소 2개짜리 리스트**입니다.

- `payload[0]` — 메타정보 (전체 건수, 페이지 수, 최종 갱신일)
- `payload[1]` — 실제 관측치 목록

문서만 읽고 `payload["data"]`로 접근하는 코드를 짜면 바로 깨집니다. **호출 한 번 해보고 구조를 확인한 뒤에 코드를 짠다** — 오늘 계속 반복할 원칙입니다.

In [ ]:
meta = payload[0]
rows = payload[1]

print("[메타]")
print(json.dumps(meta, indent=2, ensure_ascii=False))

print("\n[관측치 첫 건]")
print(json.dumps(rows[0], indent=2, ensure_ascii=False))

print(f"\n관측치 {len(rows)}건, 메타의 total은 {meta.get('total')}건")

### 메타의 `total`을 반드시 확인하세요

`total`이 `per_page`보다 크면 **뒷장이 남아 있다**는 뜻입니다. 이걸 놓치면 데이터의 앞부분만 받고 분석에 들어갑니다.

조용히 틀리는 대표적인 경우고, 오늘 오후에 이걸 자동으로 잡는 코드를 만듭니다.

In [ ]:
# 표로 펴기
df = pd.json_normalize(rows)
print("원본 컬럼:", list(df.columns))
df.head()

In [ ]:
# 필요한 것만 골라서 정리
tidy = pd.DataFrame({
    "country": df["countryiso3code"],
    "indicator": df["indicator.id"],
    "period": df["date"].astype(int),
    "value": df["value"],
}).sort_values("period").reset_index(drop=True)

tidy

### 결측을 어떻게 다룰지가 갈림길입니다

`value`에 `None`이 섞여 있습니다. 여기서 그냥 `dropna()`를 하면 정보가 사라집니다.

정책연구에서는 결측이 세 종류입니다.

| 종류 | 의미 | 분석에서 |
|---|---|---|
| 미조사 | 조사를 안 했음 | 보간 가능 여부 판단 필요 |
| 해당없음 | 개념상 존재하지 않음 | 보간하면 안 됨 |
| 비공개 | 값은 있으나 공표 안 함 | 별도 표기 필요 |

이 셋을 빈칸 하나로 뭉개면 나중에 복원할 수 없습니다. 오후에 이걸 구분해서 담는 스키마를 만듭니다.

In [ ]:
print("전체", len(tidy), "행 / 결측", tidy["value"].isna().sum(), "행")
tidy[tidy["value"].isna()]

## 1-2. 여러 국가를 한 번에

국가 코드를 세미콜론으로 이으면 한 번의 요청으로 여러 나라를 받습니다. 요청 수를 줄이는 건 호출 제한을 지키는 첫 번째 방법입니다.

In [ ]:
COUNTRIES = "KOR;JPN;FRA;DEU;ITA;ESP;USA"

payload_multi = wb_call(country=COUNTRIES, start=2015, end=2025)
rows_multi = payload_multi[1]

dfm = pd.json_normalize(rows_multi)
panel = pd.DataFrame({
    "country": dfm["countryiso3code"],
    "period": dfm["date"].astype(int),
    "value": dfm["value"],
}).dropna(subset=["value"])

# 국가 x 연도 형태로 펼쳐서 눈으로 확인
panel.pivot_table(index="period", columns="country", values="value").tail(8)

## 1-3. 국내 소스는 사정이 다릅니다

KOSIS 공유서비스와 공공데이터포털은 **인증키가 필요**합니다. 발급에 승인 대기가 있어서 당장 호출은 어려울 수 있습니다.

지금은 요청 주소가 어떻게 조립되는지만 확인하고, 실제 호출은 2교시에 명세서를 만든 뒤에 합니다.

In [ ]:
KOSIS_KEY = ""   # 발급받으셨다면 여기에 넣으세요

def kosis_url(org_id, tbl_id, start, end, key=None):
    """KOSIS 공유서비스 통계자료 요청 주소를 조립합니다. 호출은 하지 않습니다."""
    base = "https://kosis.kr/openapi/Param/statisticsParameterData.do"
    params = {
        "method": "getList",
        "apiKey": key or "<발급받은_인증키>",
        "orgId": org_id,
        "tblId": tbl_id,
        "itmId": "T10+",
        "objL1": "ALL",
        "prdSe": "Y",
        "startPrdDe": start,
        "endPrdDe": end,
        "format": "json",
        "jsonVD": "Y",
    }
    return base + "?" + "&".join(f"{k}={v}" for k, v in params.items())


print(kosis_url("101", "DT_1B81A17", "2015", "2025", KOSIS_KEY or None))

if not KOSIS_KEY:
    print("\n인증키가 없어 호출은 생략합니다. 2교시에 다시 씁니다.")

### 두 소스를 비교해 두세요

| | World Bank | KOSIS |
|---|---|---|
| 인증 | 없음 | 인증키 필요 |
| 지역 단위 | 국가 | 전국·시도·시군구 |
| 응답 최상위 | 리스트 2원소 | 딕셔너리 |
| 페이지 처리 | `page` / `per_page` | `prdSe` 등 조건으로 분할 |
| 결측 표기 | `null` | 문자 코드 |

같은 "표 하나 받기"인데 다루는 방식이 전혀 다릅니다. 소스가 늘어날수록 이 차이를 사람이 외우고 있을 수 없습니다. **그래서 명세서를 문서로 만들어 둡니다.** 2교시 주제입니다.

## 1-4. 우리 기관 소스 점검

각자 평소 쓰시는 소스 3개를 아래 표에 채워보세요. 채우다 막히는 칸이 나오면, 그게 자동화가 막히는 지점입니다.

In [ ]:
checklist = pd.DataFrame({
    "소스": ["예) KOSIS 인구동향조사", "", "", ""],
    "API 있음": ["예", "", "", ""],
    "인증 필요": ["예", "", "", ""],
    "갱신 주기": ["월", "", "", ""],
    "갱신하는 데 걸리는 시간": ["반나절", "", "", ""],
    "잠정치 있음": ["예", "", "", ""],
})
checklist

**마지막 두 칸이 오늘의 목표치입니다.**

"갱신하는 데 걸리는 시간"이 시간 단위로 적히는 소스가 있다면, 오후에 그걸 분 단위로 줄입니다.

"잠정치 있음"이 예라면, 확정치로 바뀔 때 과거 분석이 흔들립니다. 이것도 오후에 다룹니다.

---

**정리**

- 문서만 읽고 코드를 짜면 깨집니다. 호출 한 번 해보고 구조를 확인한 뒤에 짭니다
- 메타의 `total`을 확인하지 않으면 데이터 앞부분만 받고 분석에 들어갑니다
- 결측은 세 종류입니다. 한 칸으로 뭉개면 복원할 수 없습니다
- 소스마다 다루는 방식이 달라 사람이 외울 수 없습니다. 그래서 명세서를 만듭니다

다음 시간에는 API 문서를 언어모델에게 읽히고, 그 결과를 실제 호출로 검증합니다.